<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 I. RAG moderne avec LCEL — modèles OpenAI
</h1>

La clé OpenAI est chargée depuis le fichier `.env` à la racine du projet. Elle doit être définie sous le nom `OPENAI_API_KEY`.

In [ ]:
import sys
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Permet d'exécuter le notebook depuis la racine du projet ou depuis notebooks/.
racine_projet = Path.cwd()
if not (racine_projet / "data").is_dir() and (racine_projet.parent / "data").is_dir():
    racine_projet = racine_projet.parent
sys.path.insert(0, str(racine_projet / "notebooks"))



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 1 — Utiliser WordIngestion pour préparer les chunks
</h2>


In [ ]:
from WordIngestion import WordIngestion

ingestor = WordIngestion(chunk_size=500, chunk_overlap=80)
word_path = racine_projet / "data/word_files/Document_RAG_Word.docx"
chunks = ingestor.process(word_path)
print("Nombre de chunks :", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print("=" * 80)
    print(f"CHUNK {i} — source : {chunk.metadata.get('source')}")
    print(chunk.page_content[:500], "\n")



<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 2 — Créer les embeddings + Vector Store (Chroma)
</h2>


In [ ]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Charge OPENAI_API_KEY depuis le .env à la racine du projet.
load_dotenv(dotenv_path=racine_projet / ".env")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name="rag_word_docs_openai",
    persist_directory=str(racine_projet / "chroma_word_store_openai"),
    embedding_function=embeddings,
)
vector_store.add_documents(chunks, ids=[f"word-v1-{i}" for i in range(len(chunks))])
print("Index vectoriel OpenAI initialisé")

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 3 — Créer le Retriever
</h2>


In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [ ]:
retriever.invoke("Quels sont les défis liés à l'extraction du texte Word ?")

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Définir le Prompt + le pipeline LCEL
</h2>


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def format_docs(docs):
    return "\n\n".join(
        f"[Source : {doc.metadata.get('source', 'inconnue')}]\n{doc.page_content}"
        for doc in docs
    )

prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert des systèmes RAG.
Réponds uniquement à partir du contexte fourni ci-dessous et cite la source utilisée.
Si l'information n'est pas présente, réponds : « Ce n'est pas précisé dans le document. »

Contexte :
{context}

Question :
{question}

Réponse :
""")

rag_chain = (
    {
        "context": RunnableLambda(lambda x: retriever.invoke(x["question"])) | RunnableLambda(format_docs),
        "question": lambda x: x["question"],
    }
    | prompt
    | llm
    | StrOutputParser()
)

<h2 style="color:#FF8C42; font-weight:bold;">
  Étape 4 — Execution : Poser une question sur le document Word
</h2>


<h3 style="color:#C0392B; font-weight:bold;">
   Ingestion Word -> nettoyage -> chunking -> indexation Chroma -> retriever -> pipeline LCEL


In [ ]:
# 1 ) Définir la question

question = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)
question

In [ ]:
# 2) Appeler la chaîne LCEL

response = rag_chain.invoke({"question":question})

In [ ]:
# Affichage de la réponse 
print("QUESTION :")
print(question)
print("\nRéponse :")
print(response)


<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 II. Ingestion Dynamique
</h1>

In [ ]:
# Ingestion du second document Word
ingestor = WordIngestion(chunk_size=500, chunk_overlap=80)
word_v2_path = racine_projet / "data/word_files/Document_RAG_Word_v2.docx"
new_chunks = ingestor.process(word_v2_path)

print("Nombre de nouveaux chunks :", len(new_chunks))
for i, chunk in enumerate(new_chunks[:3], start=1):
    print("=" * 80)
    print(f"NOUVEAU CHUNK {i} — source : {chunk.metadata.get('source')}")
    print(chunk.page_content[:500], "\n")

In [ ]:
# Les sources sont déjà présentes dans les métadonnées produites par WordIngestion.
vector_store.add_documents(
    new_chunks,
    ids=[f"word-v2-{i}" for i in range(len(new_chunks))],
)
print("Nouveaux documents ajoutés au vector store")
print("Nombre total de documents indexés :", vector_store._collection.count())

In [ ]:
question_new = (
    "Quels sont les principaux défis liés à l'extraction de texte "
    "dans un document Word pour un pipeline RAG ?"
)
response_new = rag_chain.invoke({"question": question_new})
print("QUESTION :", question_new)
print("\nRéponse :")
print(response_new)

<h1 style="color:#D4A017; font-weight:900; font-size:34px;">
  🟨 III. Techniques avancées RAG : Mémoire conversationnelle + Vector Store
</h1>

In [ ]:
# Imports pour le RAG conversationnel
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory

# La variable embeddings a été initialisée avec OpenAIEmbeddings à l'étape précédente.
def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[Source : {doc.metadata.get('source', 'inconnue')}]\n{doc.page_content}"
        for doc in docs
    )

format_docs_runnable = RunnableLambda(format_docs)

In [ ]:
# Charger et découper le 
pdf_path = racine_projet / "data/pdf_files/rapport-activite_credit_mutuelle.pdf"
pdf_docs = PyPDFLoader(str(pdf_path)).load()
pdf_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=150,
    separators=["\n\n", "\n", ".", " ", ""],
)
chunks_pdf = pdf_splitter.split_documents(pdf_docs)
print("Nombre de chunks chargés :", len(chunks_pdf))
if chunks_pdf:
    print("Aperçu du premier chunk :", chunks_pdf[0].page_content[:500])

In [ ]:
# Utiliser les embeddings OpenAI déjà initialisés et indexer le PDF
vector_store_pdf = Chroma(
    collection_name="rapport_mutuelle_openai",
    persist_directory=str(racine_projet / "chroma_pdf_store_openai"),
    embedding_function=embeddings,
)
vector_store_pdf.add_documents(
    chunks_pdf,
    ids=[f"rapport-mutuelle-{i}" for i in range(len(chunks_pdf))],
)
print("Index PDF en base :", vector_store_pdf._collection.count())

In [ ]:
retriever_pdf = vector_store_pdf.as_retriever(search_kwargs={"k": 3})
retriever_pdf

In [ ]:
prompt_conv = ChatPromptTemplate.from_messages([
    ("system",
     "Tu es un assistant qui analyse le rapport d'activité du Crédit Mutuel. "
     "Appuie-toi uniquement sur les extraits fournis, cite les sources et réponds clairement en français. "
     "Si les extraits ne suffisent pas, dis que tu ne sais pas."),
    MessagesPlaceholder("history"),
    ("human", "Extraits pertinents :\n\n{context}\n\nQuestion : {question}"),
])

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag_chain_pdf = (
    {
        "question": itemgetter("question"),
        "context": itemgetter("question") | retriever_pdf | format_docs_runnable,
        "history": itemgetter("history"),
    }
    | prompt_conv
    | llm
    | StrOutputParser()
)

In [ ]:
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

rag_with_memory = RunnableWithMessageHistory(
    rag_chain_pdf,
    get_session_history,
    input_messages_key="question",
    history_messages_key="history",
)

In [ ]:
session_id = "mutuelle_session_1"

def ask(question):
    response = rag_with_memory.invoke(
        {"question": question},
        config={"configurable": {"session_id": session_id}},
    )
    print(f"\n❓ {question}\n")
    print("💬 Réponse :\n", response)
    print("\n" + "=" * 80)

ask("Quels sont les grands axes stratégiques du rapport d'activité ?")
ask("Peux-tu résumer ces axes en mettant l'accent sur la transformation digitale ?")
ask("Comment ces axes stratégiques influencent-ils la relation sociétaire ?")
ask("Et résume-moi tout cela en une phrase comme si tu parlais à un sociétaire.")

In [ ]:
# Tests supplémentaires du RAG conversationnel (même session)
ask("Peux-tu rappeler les principaux axes stratégiques ?")

In [ ]:
ask("Résume les éléments essentiels en une phrase.")